# NIG Project : Market Data Collection and Cleaning

This notebook prepares the SPX European call-option dataset used in the NIG calibration workflow.

## Objective and data-selection methodology

**Observation date:** 23 October 2026  
**Underlying:** S&P 500 Index (SPX)  
**Option type:** European calls  
**Expiry buckets:** five  
**Initial moneyness range:** (0.90 \leq K/S_0 \leq 1.10)

The workflow imports the Bloomberg quotes, selects contracts within the stated moneyness range, constructs midpoint prices, checks quote quality and market consistency, attaches market inputs, and retains contracts with positive trading volume.

The midpoint ((Bid+Ask)/2) is used as the market-price target, in line with the agreed project convention. The last traded price is retained only as a source field and is not used as the calibration target.

The final output is a long-format dataset: one row per option contract, with the market and model inputs required by the calibration code.

## 1. Imports

Load the data-analysis libraries used throughout the notebook.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

## 2. Load the Bloomberg workbook

Read each expiry sheet using the common Bloomberg column layout. Empty rows are removed before any filtering is performed.

In [ ]:
# Locate the project root and the original Bloomberg workbook.
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "data").is_dir() else cwd.parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

# Accept the standard filename and the filename used in the uploaded source file.
raw_candidates = [
    RAW_DATA_DIR / "SPX_options_data.xlsx",
]
FILE_PATH = next((candidate for candidate in raw_candidates if candidate.exists()), None)

if FILE_PATH is None:
    expected_files = "\\n".join(str(candidate) for candidate in raw_candidates)
    raise FileNotFoundError(
        "Bloomberg source workbook not found. Place the original workbook in "
        f"{RAW_DATA_DIR} using one of these filenames:\\n{expected_files}"
    )

COLUMN_NAMES = [
    "Strike",
    "Ticker",
    "Bid",
    "Ask",
    "Last Trade Price",
    "Implied Volatility",
    "Trading Volume",
]

all_raw = pd.read_excel(
    FILE_PATH,
    sheet_name=None,
    skiprows=2,
    header=None,
    names=COLUMN_NAMES,
)

# Remove fully empty rows from each expiry sheet.
for sheet in all_raw.values():
    sheet.dropna(how="all", inplace=True)

print("Loaded source workbook:", FILE_PATH)

Inspect the imported sheets to confirm that all five expiries have been loaded and that the columns are aligned as expected.

In [ ]:
# Inspect the size and structure of each imported expiry sheet.
for sheet_name, sheet in all_raw.items():
    print(sheet_name, sheet.shape)
    display(sheet.head())

## 3. Define the initial moneyness universe

Restrict the strike range to 90%-110% of the observation-date spot level. This focuses the analysis on contracts reasonably close to at-the-money while retaining both in-the-money and out-of-the-money observations.

In [ ]:
# Keep calls with strikes within 90%–110% of the SPX spot level.
S_0 = 7623.68

df = {
    name: sheet.loc[
        sheet["Strike"].between(0.90 * S_0, 1.10 * S_0)
    ].copy().reset_index(drop=True)
    for name, sheet in all_raw.items()
}

for sheet_name, sheet in df.items():
    print(sheet_name, sheet.shape)
    display(sheet.head())

## 4. Quote inspection and midpoint construction

Check zero bid and ask quotes, then compute the bid-ask midpoint. The midpoint is the market-price observation used in the calibration workflow.

In [ ]:
# Inspect quote validity and construct the bid-ask midpoint.
for sheet_name, sheet in df.items():
    print(f"{sheet_name}: zero bids = {(sheet['Bid'] == 0).sum()}")
    print(f"{sheet_name}: zero asks = {(sheet['Ask'] == 0).sum()}")

    # The midpoint is the market-price input used in the calibration.
    sheet["Mid"] = (sheet["Bid"] + sheet["Ask"]) / 2
    sheet["Market_Price"] = sheet["Mid"]

for sheet_name, sheet in df.items():
    print(sheet_name, sheet.shape)
    display(sheet.head())

## 5. Quote-quality checks

Review missing fields, non-positive quotes, crossed or locked markets, duplicate tickers, trading-volume values and implied-volatility values. These diagnostics help identify data issues before the final sample is assembled.

In [ ]:
# Summarise quote integrity separately for each expiry.
quality_results = []

for expiry, sheet in df.items():
    quality_results.append({
        "Expiry": expiry,
        "Number of Quotes": len(sheet),
        "Missing Strike": sheet["Strike"].isna().sum(),
        "Missing Ticker": sheet["Ticker"].isna().sum(),
        "Missing Bid": sheet["Bid"].isna().sum(),
        "Missing Ask": sheet["Ask"].isna().sum(),
        "Missing Mid": sheet["Mid"].isna().sum(),
        "Non-positive Bid": (sheet["Bid"] <= 0).sum(),
        "Non-positive Ask": (sheet["Ask"] <= 0).sum(),
        "Crossed Quotes": (sheet["Ask"] < sheet["Bid"]).sum(),
        "Locked Quotes": (sheet["Ask"] == sheet["Bid"]).sum(),
        "Duplicate Tickers": sheet["Ticker"].duplicated().sum(),
        "Negative Volume": (sheet["Trading Volume"] < 0).sum(),
        "Zero Volume": (sheet["Trading Volume"] == 0).sum(),
        "Positive Volume": (sheet["Trading Volume"] > 0).sum(),
        "Zero IV": (sheet["Implied Volatility"] == 0).sum(),
        "Negative IV": (sheet["Implied Volatility"] < 0).sum(),
        "Missing IV": sheet["Implied Volatility"].isna().sum(),
    })

quality_report = pd.DataFrame(quality_results)
display(quality_report)

## 6. Bid-ask spread diagnostics

Summarise absolute and relative bid-ask spreads by expiry. This section is descriptive: it provides context on quote dispersion and is not used as a contract-selection rule.

In [ ]:
# Describe quoted bid-ask spreads as a market-liquidity diagnostic.
liquidity_results = []

for expiry, sheet in df.items():
    sheet["Absolute_Spread"] = sheet["Ask"] - sheet["Bid"]
    sheet["Relative_Spread"] = (
        sheet["Absolute_Spread"] / sheet["Mid"]
    ) * 100

    liquidity_results.append({
        "Expiry": expiry,
        "Number of Quotes": len(sheet),
        "Median Absolute Spread": sheet["Absolute_Spread"].median(),
        "Median Relative Spread (%)": sheet["Relative_Spread"].median(),
        "75th Percentile (%)": sheet["Relative_Spread"].quantile(0.75),
        "90th Percentile (%)": sheet["Relative_Spread"].quantile(0.90),
        "Positive Volume": (sheet["Trading Volume"] > 0).sum(),
        "Zero Volume": (sheet["Trading Volume"] == 0).sum(),
    })

liquidity_report = pd.DataFrame(liquidity_results)
display(liquidity_report.round(4))

## 7. Maturity and day-count convention

Verify the calendar days between the observation date and each expiry. The time to maturity is expressed on ACT/365, while the quoted short-term OIS rates are first converted using ACT/360.

In [ ]:
from datetime import datetime

# Define the observation date and the five option expiries.
valuation_date = datetime(2026, 10, 23)
expiry_dates = [
    datetime(2026, 11, 20),
    datetime(2026, 12, 18),
    datetime(2027, 1, 15),
    datetime(2027, 4, 16),
    datetime(2027, 9, 17),
]
maturities_days = [28, 56, 84, 175, 329]

actual_days = [(expiry - valuation_date).days for expiry in expiry_dates]

maturity_check = pd.DataFrame({
    "Expiry": list(df.keys()),
    "Expiry Date": expiry_dates,
    "Expected Days": maturities_days,
    "Actual Calendar Days": actual_days,
})
maturity_check["Match"] = (
    maturity_check["Expected Days"] == maturity_check["Actual Calendar Days"]
)
display(maturity_check)

## 8. Market inputs and European lower-bound checks

Associate each expiry with its annual dividend yield and OIS rate. The quoted OIS rates are simple ACT/360 rates: convert each one into a discount factor, then derive its equivalent continuously compounded ACT/365 rate.

The quoted annual dividend yield is 1.06%. Convert it into the continuously compounded convention used by the risk-neutral pricing formula:

\[
q_{\\mathrm{cont}}=\\ln(1+q_{\\mathrm{quoted}}).
\]

Finally, compare midpoint prices with the European call lower bound and the additional bound reported in the project references.

In [ ]:
# Market inputs observed for the five maturity buckets.
q = 1.06  # Annual dividend yield, expressed in percent.
risk_free_rates = [3.93, 4.03, 4.08, 4.20, 4.37]

In [ ]:
# Convert quoted OIS rates and the quoted dividend yield to continuous rates.
q_cont = np.log1p(q / 100)

for (name, sheet), rate_pct, days in zip(df.items(), risk_free_rates, maturities_days):
    tau = days / 365
    discount = 1 / (1 + (rate_pct / 100) * days / 360)

    sheet["Days"] = days
    sheet["tau"] = tau
    sheet["Discount"] = discount
    sheet["r"] = -np.log(discount) / tau

    sheet["LB_European"] = np.maximum(
        0,
        S_0 * np.exp(-q_cont * tau) - sheet["Strike"] * discount,
    )
    sheet["LB_BCC"] = np.maximum(
        sheet["LB_European"],
        S_0 - sheet["Strike"],
    )
    sheet["Violates_European"] = (
        sheet["Market_Price"] < sheet["LB_European"]
    )
    sheet["Violates_BCC"] = (
        sheet["Market_Price"] < sheet["LB_BCC"]
    )

    print(
        name,
        "European lower-bound violations:",
        int(sheet["Violates_European"].sum()),
        "| BCC lower-bound violations:",
        int(sheet["Violates_BCC"].sum()),
    )

## 9. Build the long-format option table

Combine the five expiry sheets into a single table. Each row corresponds to one contract and retains its expiry label, strike, quote fields, midpoint and market inputs.

In [ ]:
# Combine all expiry sheets into one long-format table.
# Each row represents one SPX call contract.
options = pd.concat(
    [sheet.assign(Expiry=expiry) for expiry, sheet in df.items()],
    ignore_index=True,
)
options = options.sort_values(["Expiry", "Strike"]).reset_index(drop=True)

print("Total contracts in the moneyness universe:", len(options))
print("Number of expiries:", options["Expiry"].nunique())
display(options.head())

## 10. Monotonicity and convexity diagnostics

For each expiry, sort calls by strike and inspect whether midpoint prices are non-increasing and whether successive price slopes are non-decreasing. These are diagnostic checks on the observed price curve.

In [ ]:
# Check monotonicity and convexity of call prices within each expiry.
convexity_results = []

for expiry, group in options.groupby("Expiry"):
    group = group.sort_values("Strike")
    strikes = group["Strike"].to_numpy()
    prices = group["Market_Price"].to_numpy()

    price_diff = np.diff(prices)
    strike_diff = np.diff(strikes)
    slopes = price_diff / strike_diff

    convexity_results.append({
        "Expiry": expiry,
        "Number of Options": len(group),
        "Monotonicity Violations": int(np.sum(price_diff > 0)),
        "Convexity Violations": int(np.sum(np.diff(slopes) < 0)),
    })

display(pd.DataFrame(convexity_results))

Identify the individual strike triplets associated with any midpoint convexity violations. This provides contract-level detail for the summary above.

In [ ]:
# Locate individual strike triplets where midpoint prices violate convexity.
convexity_details = []

for expiry, group in options.groupby("Expiry"):
    group = group.sort_values("Strike").reset_index(drop=True)
    strikes = group["Strike"].to_numpy()
    prices = group["Market_Price"].to_numpy()
    slopes = np.diff(prices) / np.diff(strikes)

    for i in range(len(slopes) - 1):
        if slopes[i + 1] < slopes[i]:
            convexity_details.append({
                "Expiry": expiry,
                "K1": strikes[i],
                "K2": strikes[i + 1],
                "K3": strikes[i + 2],
                "C1": prices[i],
                "C2": prices[i + 1],
                "C3": prices[i + 2],
                "Slope 1": slopes[i],
                "Slope 2": slopes[i + 1],
            })

display(pd.DataFrame(convexity_details))

Check whether the bid-ask intervals can accommodate a convex selection of prices for consecutive strike triplets. This complements the midpoint-based convexity diagnostic.

In [ ]:
# Check whether bid-ask intervals can accommodate a convex price selection.
interval_results = []

for expiry, group in options.groupby("Expiry"):
    group = group.sort_values("Strike").reset_index(drop=True)
    strikes = group["Strike"].to_numpy()
    bids = group["Bid"].to_numpy()
    asks = group["Ask"].to_numpy()

    h1 = np.diff(strikes)[:-1]
    h2 = np.diff(strikes)[1:]

    max_slope_difference = (
        asks[2:] / h2
        + asks[:-2] / h1
        - bids[1:-1] * (1 / h1 + 1 / h2)
    )
    compatible = max_slope_difference >= 0

    interval_results.append({
        "Expiry": expiry,
        "Total Triplets": len(compatible),
        "Bid-Ask Compatible Triplets": int(compatible.sum()),
        "Incompatible Triplets": int((~compatible).sum()),
    })

display(pd.DataFrame(interval_results))

## 11. Strike coverage across expiries

Identify strikes quoted at all five expiries and summarise the overlap. This is a coverage diagnostic and does not change the selected calibration sample.

In [ ]:
# Measure strike overlap across the five expiries.
maturities = sorted(options["Expiry"].unique())

strike_coverage = (
    options.groupby("Strike")["Expiry"]
    .nunique()
    .reset_index(name="Number_of_Expiries")
)
common_strikes = strike_coverage.loc[
    strike_coverage["Number_of_Expiries"] == len(maturities),
    "Strike",
]

common_strike_summary = (
    options[options["Strike"].isin(common_strikes)]
    .groupby("Expiry")
    .agg(
        Number_of_Common_Strikes=("Strike", "nunique"),
        Number_of_Quotes=("Strike", "count"),
    )
    .reset_index()
)

print("Number of expiries:", len(maturities))
print("Number of strikes common to all expiries:", len(common_strikes))
display(common_strike_summary)

## 12. Construct the calibration sample

Retain contracts with positive observed trading volume. This produces the 477-observation sample used as the primary calibration candidate set. No additional spread-based selection rule is applied.

In [ ]:
# Select contracts with observed trading activity.
# This is the sole sample-selection criterion applied after the moneyness universe.
liquid_options = options.loc[options["Trading Volume"] > 0].copy()
liquid_options = liquid_options.reset_index(drop=True)

print("Moneyness-universe contracts:", len(options))
print("Positive-volume contracts:", len(liquid_options))
print("Number of expiries retained:", liquid_options["Expiry"].nunique())

sample_comparison = pd.DataFrame({
    "Moneyness Universe": options.groupby("Expiry").size(),
    "Positive-Volume Sample": liquid_options.groupby("Expiry").size(),
})
display(sample_comparison)

Describe the spread distribution within the selected sample. These statistics remain descriptive and do not determine whether an observation is retained.

In [ ]:
# Summarise the quoted bid-ask spread distribution within the selected sample.
display(
    liquid_options.groupby("Expiry")["Relative_Spread"].agg(
        Count="count",
        Median="median",
        Mean="mean",
        Std="std",
        P75=lambda x: x.quantile(0.75),
        P90=lambda x: x.quantile(0.90),
        Maximum="max",
    )
)

Review the strike and moneyness range represented in the selected sample for each expiry.

In [ ]:
# Review strike and moneyness coverage after the positive-volume selection.
liquid_options["Moneyness"] = liquid_options["Strike"] / S_0

coverage_summary = (
    liquid_options.groupby("Expiry")
    .agg(
        Number_of_Contracts=("Ticker", "count"),
        Minimum_Strike=("Strike", "min"),
        Maximum_Strike=("Strike", "max"),
        Minimum_Moneyness=("Moneyness", "min"),
        Maximum_Moneyness=("Moneyness", "max"),
        Unique_Strikes=("Strike", "nunique"),
    )
)
display(coverage_summary)

Group the selected contracts into moneyness buckets to visualise the distribution of strikes across expiries.

In [ ]:
# Describe the moneyness composition of the selected contracts.
bins = [0.90, 0.95, 1.00, 1.05, 1.10]
labels = [
    "ITM (0.90–0.95)",
    "ITM / Near ATM (0.95–1.00)",
    "OTM / Near ATM (1.00–1.05)",
    "OTM (1.05–1.10)",
]

liquid_options["Moneyness_Bucket"] = pd.cut(
    liquid_options["Moneyness"],
    bins=bins,
    labels=labels,
    include_lowest=True,
)
moneyness_distribution = pd.crosstab(
    liquid_options["Expiry"],
    liquid_options["Moneyness_Bucket"],
)
display(moneyness_distribution)

## 13. Assemble and export the calibration dataset

Attach the common spot, dividend yield, maturity, continuously compounded risk-free rate and midpoint market price. Rename the fields to the canonical schema used by the validation notebook and calibration module.

In [ ]:
# Assemble the selected observations and attach the inputs required by calibration.
calibration_data = liquid_options.copy()

calibration_data["Spot"] = S_0
calibration_data["Dividend_Yield"] = q_cont
calibration_data["Market_Price"] = calibration_data["Mid"]

calibration_data = calibration_data.rename(
    columns={
        "tau": "Tau",
        "r": "Risk_Free_Rate_CC",
        "Trading Volume": "Trading_Volume",
        "Implied Volatility": "Implied_Volatility",
    }
)
calibration_data["Implied_Volatility"] = (
    calibration_data["Implied_Volatility"] / 100
)
calibration_data = calibration_data.rename(columns={"Days": "Days_To_Maturity"})

print("Number of observations:", len(calibration_data))
print("Number of expiries:", calibration_data["Expiry"].nunique())
print("Continuous dividend yield:", q_cont)
print("Missing values:", calibration_data.isna().sum().sum())
display(calibration_data.head())

Inspect the final columns, data types and descriptive statistics before exporting the dataset.

In [ ]:
# Inspect the final calibration dataset and its descriptive statistics.
print("Available columns:")
print(calibration_data.columns.tolist())
calibration_data.info()
display(calibration_data.describe().T)

Select the fields required by the downstream workflow and save the compressed CSV in `data/cleaned/`. The same file is read by the data-validation notebook and the calibration script.

In [ ]:
# Keep the canonical fields consumed by the validation and calibration workflow.
calibration_columns = [
    "Expiry",
    "Days_To_Maturity",
    "Strike",
    "Market_Price",
    "Spot",
    "Tau",
    "Risk_Free_Rate_CC",
    "Dividend_Yield",
    "Trading_Volume",
    "Implied_Volatility",
    "Relative_Spread",
]
calibration_data = calibration_data[calibration_columns].copy()

# Export to the project data directory, independent of the notebook working directory.
OUTPUT_PATH = PROJECT_ROOT / "data" / "cleaned" / "calibration_candidates.csv.gz"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
calibration_data.to_csv(OUTPUT_PATH, index=False, compression="gzip")

print("Final dataset shape:", calibration_data.shape)
print("Saved to:", OUTPUT_PATH)
display(calibration_data.head())